In [11]:
from dotenv import load_dotenv
from openai import OpenAI
from  PyPDF2 import PdfReader
import gradio as gr

In [12]:
load_dotenv(override=True)
openai = OpenAI()

In [13]:
reader = PdfReader("me/resume.pdf")
resume = ""

for  page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [14]:
print (resume)

 FARUK HOSSAIN  Email:  miloniitju@gmail.com 
 Kuala Lumpur, Malaysia  Mobile: +60 111-667-5713 
 emotionless  emotionless  faruk 
 Work Experience 
 Technical Lead:  PETRONAS Digital Sdn Bhd  , Kuala Lumpur,  Malaysia  April 2024 - Current 
 Project Name:  Datathon 
 ●  Led  architecture  and  execution  of  2TB+  RDS-to-Databricks  migration,  improving  analytics  speed  by  40%  and  reducing 
 infra cost by  25%  . 
 ●  Directed design of scalable ETL pipelines processing  100K+  daily records with zero data loss and full  schema integrity. 
 ●  Enabled  near  real-time  analytics  by  integrating  streaming  data  sources  into  Databricks,  reducing  reporting  lag  from 
 hours  to under  10  minutes 
 ●  Mentored  4  engineers  and drove cross-team alignment  to deliver on-time migration with  100%  SLA compliance. 
 ●  Tools & Technologies: Python, PySpark, Delta Lake, AWS, DBC Workflow, Medallion Architecture, Azure DevOps 
 Project Name:  Heatmap 
 ●  Led a  7-engineer  tea

In [15]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()



In [16]:
name = "Faruk"

In [17]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [18]:
print (system_prompt)

You are acting as Faruk. You are answering questions on Faruk's website, particularly questions related to Faruk's career, background, skills and experience. Your responsibility is to represent Faruk for interactions on the website as faithfully as possible. You are given a summary of Faruk's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.

## Summary:
I am Faruk, and I am originally from Gazipur, Bangladesh.
Currently I live in Malaysia. Apart from that, previously I lived in Canada, Bangladesh, South Korea.
I love malay food very much (Nasi  lemak is my favourite).

## Resume:
 FARUK HOSSAIN  Email:  miloniitju@gmail.com 
 Kuala Lumpur, Malaysia  Mobile: +60 111-667-5713 
 emotionless  emotionless  faruk 
 Work Experience 
 Technical Lead:  PETRONAS Digital Sdn Bhd  , Kuala Lumpur,  Malaysia  April 2024 - Curren

In [19]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role":  "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content


In [20]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [21]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [22]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"


evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [24]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return  user_prompt


In [25]:
def evaluate(reply, message, history):
    messages = [{"role": "system", "content": evaluator_system_prompt}]  + [{"role":  "user", "content": evaluator_user_prompt}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}]  + [{"role":  "user", "content": "Did you have a good problem solving skill?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
reply = response.choices[0].message.content

TypeError: You tried to pass a `BaseModel` class to `chat.completions.create()`; You must use `chat.completions.parse()` instead